In [3]:
!nvidia-smi

%cd /content
!git clone https://github.com/nuhaminae/The-Conversion-Engine-Ground-Truth
%cd /content/The-Conversion-Engine-Ground-Truth

Sat May  2 16:56:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
# ============================================================
# Colab setup for eval_prompted_judge.py
# ============================================================

import os
from pathlib import Path

# Must be set before importing transformers in the subprocess.
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"
os.environ["USE_TORCHVISION"] = "0"
os.environ["WANDB_DISABLED"] = "true"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

REPO = Path("/content/The-Conversion-Engine-Ground-Truth")
assert REPO.exists(), f"Repo not found at {REPO}. Clone/pull the repo first."
os.chdir(REPO)
print("Working directory:", os.getcwd())

!nvidia-smi

# Minimal stack for prompted judge evaluation on Colab.
!pip install -q --upgrade pip
!pip install -q "transformers==4.56.2" "accelerate" "bitsandbytes" "datasets" "pyyaml" "python-dotenv" "scikit-learn" "matplotlib" "sentencepiece" "hf_transfer"

Working directory: /content/The-Conversion-Engine-Ground-Truth
Sat May  2 16:56:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |         

In [5]:
# ============================================================
# Run prompted judge evaluation
# ============================================================

import os
from pathlib import Path
from getpass import getpass
import yaml

# Environment flags inherited by the subprocess.
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"
os.environ["USE_TORCHVISION"] = "0"
os.environ["WANDB_DISABLED"] = "true"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

REPO = Path("/content/The-Conversion-Engine-Ground-Truth")
os.chdir(REPO)
print("Working directory:", os.getcwd())

# Optional but recommended for higher HF download limits.
hf_token = getpass("HF token, or press Enter to skip: ").strip()
os.environ["HF_TOKEN"] = hf_token
os.environ["WANDB_API_KEY"] = ""

# Make sure prompted judge is configured for Colab/T4.
config_path = Path("configs/eval_config.yaml")
assert config_path.exists(), "Missing configs/eval_config.yaml"

with config_path.open("r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

config.setdefault("prompted_judge", {})
config["prompted_judge"]["base_model"] = config["prompted_judge"].get(
    "base_model",
    "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
)
config["prompted_judge"]["tokenizer_name"] = config["prompted_judge"].get(
    "tokenizer_name",
    config["prompted_judge"]["base_model"],
)
config["prompted_judge"]["load_in_4bit"] = True
config["prompted_judge"]["max_length"] = int(config["prompted_judge"].get("max_length", 1024))
config["prompted_judge"]["threshold"] = float(config["prompted_judge"].get("threshold", 0.0))
config["prompted_judge"]["prompt_template_version"] = config["prompted_judge"].get(
    "prompt_template_version",
    "tenacious_prompted_judge_v1",
)

config.setdefault("data", {})
config["data"]["heldout_file"] = config["data"].get(
    "heldout_file",
    "tenacious_bench/held_out/held_out.jsonl",
)
config["data"]["heldout_dpo_file"] = config["data"].get(
    "heldout_dpo_file",
    "tenacious_bench/dpo/held_out_dpo.jsonl",
)
config["data"]["output_dir"] = config["data"].get("output_dir", "reports/evaluation")

config.setdefault("outputs", {})
config["outputs"]["prompted_judge_metrics"] = config["outputs"].get(
    "prompted_judge_metrics",
    "reports/evaluation/prompted_judge_metrics.json",
)
config["outputs"]["prompted_judge_pointwise_scores"] = config["outputs"].get(
    "prompted_judge_pointwise_scores",
    "reports/evaluation/prompted_judge_pointwise_scores.jsonl",
)
config["outputs"]["prompted_judge_pair_scores"] = config["outputs"].get(
    "prompted_judge_pair_scores",
    "reports/evaluation/prompted_judge_pair_scores.jsonl",
)

config.setdefault("logging", {})
config["logging"]["prompted_log"] = config["logging"].get(
    "prompted_log",
    "reports/evaluation/eval_prompted_judge.log",
)

config.setdefault("plots", {})
config["plots"].setdefault("confusion_matrix", {})
config["plots"]["confusion_matrix"]["prompted_judge_filename"] = config["plots"]["confusion_matrix"].get(
    "prompted_judge_filename",
    "prompted_judge_confusion_matrix.png",
)
config["plots"]["confusion_matrix"]["labels"] = config["plots"]["confusion_matrix"].get(
    "labels",
    ["Bad Output", "Good Output"],
)

with config_path.open("w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, sort_keys=False)

# Validate required files.
required_files = [
    Path("src/evaluation/eval_prompted_judge.py"),
    Path("configs/eval_config.yaml"),
    Path(config["data"]["heldout_file"]),
]

for path in required_files:
    assert path.exists(), f"Missing required file: {path}"

print("Required files exist.")
!python -m py_compile src/evaluation/eval_prompted_judge.py

# Run evaluation.
!mkdir -p reports/evaluation

!PYTHONUNBUFFERED=1 python src/evaluation/eval_prompted_judge.py \
  --config configs/eval_config.yaml \
  2>&1 | tee reports/evaluation/eval_prompted_judge.log

print("\nPrompted judge metrics:")
!cat reports/evaluation//prompted_judge_metrics.json || true

print("\nPrompted judge outputs:")
!find reports -maxdepth 2 -type f \( -name "*prompted*" -o -name "*confusion*" \) -print || true

Working directory: /content/The-Conversion-Engine-Ground-Truth
HF token, or press Enter to skip: ··········
Required files exist.
Starting prompt-engineered judge evaluation.
Loaded 52 pointwise held-out examples.
Loading prompted judge base model: unsloth/Llama-3.2-1B-Instruct-bnb-4bit
Loading tokenizer: unsloth/Llama-3.2-1B-Instruct-bnb-4bit
/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:239: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)
[1/52] pair_id=adv_cd-01_add305cb70e4 label=1 pred=0 margin=-0.5784 correct=False
[2/52] pair_id=adv_cd-01_add305cb70e4 label=0 pred=0 margin=-1.3123 correct=True
[3/52] pair_id=adv_cd-02_02c94d8859a6 label=1 pred=1 margin=0.2812 correct=True
[4/52] pair_id=adv_cd-02_02c94d8859a6 label=0 pred=0 margin=-2.3601 correct=True
[

In [6]:
!zip -r prompted_judge_eval_outputs.zip \
  reports/evaluation/prompted_judge_metrics.json \
  reports/evaluation/prompted_judge_pointwise_scores.jsonl \
  reports/evaluation/prompted_judge_pair_scores.jsonl \
  reports/evaluation/prompted_judge_confusion_matrix.png \
  reports/evaluation/eval_prompted_judge.log \
  configs/eval_config.yaml

from google.colab import files
files.download("prompted_judge_eval_outputs.zip")

  adding: reports/evaluation/prompted_judge_metrics.json (deflated 45%)
  adding: reports/evaluation/prompted_judge_pointwise_scores.jsonl (deflated 83%)
  adding: reports/evaluation/prompted_judge_pair_scores.jsonl (deflated 86%)
  adding: reports/evaluation/prompted_judge_confusion_matrix.png (deflated 15%)
  adding: reports/evaluation/eval_prompted_judge.log (deflated 76%)
  adding: configs/eval_config.yaml (deflated 68%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>